In [1]:
import brainsss
import os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import colors
%matplotlib inline
#from sklearn.cluster import AgglomerativeClustering
import scipy
import time
import h5py
import ants
import nibabel as nib
from scipy.ndimage import uniform_filter, gaussian_filter
import shutil
from sklearn.cluster import AgglomerativeClustering
from sklearn.feature_extraction.image import grid_to_graph
import gc
import sys
import traceback
sys.tracebacklimit = 1000  # Show full traceback
import warnings
from scipy.ndimage import gaussian_filter1d,gaussian_filter
from scipy.signal import butter, sosfiltfilt, filtfilt, freqz,iirnotch
import cv2
from scipy.ndimage.morphology import binary_erosion
from scipy.ndimage.morphology import binary_dilation
from scipy.ndimage import zoom
from sklearn.mixture import GaussianMixture
import scipy.stats as stats
import sklearn
import pickle
import itertools
from statsmodels.stats.multitest import multipletests
import seaborn as sns

In [10]:
def get_var_size():
    # Get all variables in the current namespace
    variables = {name: value for name, value in globals().items() if not name.startswith("__")}

    # Sort variables by size
    sorted_vars = sorted(variables.items(), key=lambda x: sys.getsizeof(x[1]), reverse=True)

    # Print variable sizes
    for name, value in sorted_vars:
        print(f"{name}: {sys.getsizeof(value) / 1024:.2f} KB")

In [3]:
### big STA supervox
fly_path = '/oak/stanford/groups/trc/data/Ilana/2P/data/'
home_path = '/oak/stanford/groups/trc/data/Ilana/2P/data/later/'
later_dir = '/oak/stanford/groups/trc/data/Ilana/2P/data/later/temp_filter'
cluster_dir = os.path.join(later_dir, 'clustering')
save_dir=os.path.join(later_dir,'figs')
# behaviors = ['inc', 'dec', 'flat']

n_vox=2000

channel=2

event='10flies_5sec'
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [4]:
if event != None:
    event_times_path = os.path.join(home_path, f'{event}_event_times_split_dic.pkl')
else:
    event_times_path = os.path.join(home_path, 'event_times_split_dic.pkl')
with open(event_times_path, 'rb') as file:
    event_times_struct = pickle.load(file)
    f=list(event_times_struct.keys())[0]
    behaviors=list(event_times_struct[f].keys())
    print(f"Found behaviors: {behaviors}")

Found behaviors: ['total', 'inc', 'dec', 'flat']


In [5]:
supercluster_labels=np.load(os.path.join(cluster_dir, 'supercluster_labels_total_500.npy'))

In [ ]:
fn=226
b='total'
fly=f'fly_{fn}'
temp_dir=os.path.join(fly_path,fly,'temp_filter')
for fx in os.listdir(temp_dir):
    if '500_' not in fx and f'_{b}_' in fx and event in fx and f'_{channel}_' in fx:
#     if '5sec' in fx and f'_{b}_' in fx:
        print(fly,fx)
        file_need=(fx)
        if file_need!=[]:
            path=os.path.join(temp_dir,file_need)
            with h5py.File(path, 'r') as hf:
                brain = hf['brain'][:]
                ts = hf['time_stamps'][:]
                dimsb = np.shape(brain)
                dimst = np.shape(ts)
                print(f"Brain shape is {dimsb}, time stamp shape is {dimst}")

In [ ]:
np.nanmax(ts)

In [ ]:
intervals=np.asarray([[-600,0],[700,1300]])

In [ ]:
steps=600
range_end=2000
range_start=-600
print(range_start,range_end)

In [ ]:
1300-700

In [ ]:
%%time
temp=[]
for interval in intervals:
#     print(interval)
    start=interval[0]
    end=interval[1]
#     end = i + steps if i + steps < range_end else range_end
    mask = (ts > start) & (ts < end)
    result = np.where(mask, brain, np.nan)
    temp.append(result)
temp=np.asarray(temp)
brain_w_ts=np.moveaxis(temp,0,-1)
brain_w_ts.shape
print(brain_w_ts.shape)

In [ ]:
%%time
super_clust=500
brain_new=brain_w_ts
neural_activity= brain_new.reshape(-1, brain_w_ts.shape[-2], brain_w_ts.shape[-1])

behavior_superclusters = []
for cluster_num in range(super_clust):
    labels= supercluster_labels
    cluster_indicies= np.where(labels==cluster_num)[0]
    mean_signal = np.mean(neural_activity[cluster_indicies,:], axis=0)
    behavior_superclusters.append(mean_signal)
behavior_superclusters = np.asarray(behavior_superclusters)

In [ ]:
fig,ax=plt.subplots()
ax.imshow(np.nanmean(behavior_superclusters,axis=-2))
ax.set_aspect(0.01)

In [ ]:
len(intervals)

In [6]:
%%time
fly_superclust_dict={'inc':{}, 'dec':{}, 'flat':{}, 'total':{}}
fly_num=[226,227,228,234,239,240,241,242,249,250]
# range_start=-2000; range_end=3000; steps=500
intervals=np.asarray([[-600,0],[700,1300]])
super_clust=500
# n_steps = len(range(range_start, range_end, steps))
n_steps=len(intervals)

for b in behaviors:
    for fn in fly_num:
        fly=f'fly_{fn}'
        temp_dir=os.path.join(fly_path,fly,'temp_filter')
        for fx in os.listdir(temp_dir):
            if '500_' not in fx and f'_{b}_' in fx and event in fx and f'_{channel}_' in fx:
                print(fly,fx)
                file_need=(fx)
                if file_need!=[]:
                    path=os.path.join(temp_dir,file_need)
                    with h5py.File(path, 'r') as hf:
                        brain = hf['brain']
                        ts = hf['time_stamps'][:]
                        dimsb = np.shape(brain)
                        dimst = np.shape(ts)
                        print(f"Brain shape is {dimsb}, time stamp shape is {dimst}")
                        temp=[]
                        for interval in intervals:
#                             start=i
#                             end = i + steps if i + steps < range_end else range_end
                            start=interval[0]; end=interval[1]
                            mask = (ts > start) & (ts < end)
                            result = np.where(mask, brain, np.nan)
                            temp.append(result)
                        temp=np.asarray(temp)
                        brain_w_ts=np.moveaxis(temp,0,-1)
                        five_shape=brain_w_ts.shape
                        print(five_shape)
                        superclust_dict = {}
                        brain_new=brain_w_ts
                        neural_activity= brain_new.reshape(-1, five_shape[-2], five_shape[-1])

                        behavior_superclusters = []
                        for cluster_num in range(super_clust):
                            labels= supercluster_labels
                            cluster_indicies= np.where(labels==cluster_num)[0]
                            mean_signal = np.mean(neural_activity[cluster_indicies,:], axis=0)
                            behavior_superclusters.append(mean_signal)
                        behavior_superclusters = np.asarray(behavior_superclusters)
                        print(behavior_superclusters.shape)
                        fly_superclust_dict[b][fn]=behavior_superclusters
                        sorted_vars=get_var_size()
                        del brain,neural_activity,behavior_superclusters,brain_new,ts,result,mask,labels,mean_signal
                        gc.collect()
                else:
                    print(f'{fly} does not contribute to this behavior')

fly_226 functional_channel_2_moco_warp_blurred_hpf_dff_filtered_total_10flies_5sec.h5
Brain shape is (314, 146, 91, 1960), time stamp shape is (314, 146, 91, 1960)
(314, 146, 91, 1960, 2)
(500, 1960, 2)
fly_227 functional_channel_2_moco_warp_blurred_hpf_dff_filtered_total_10flies_5sec.h5
Brain shape is (314, 146, 91, 1960), time stamp shape is (314, 146, 91, 1960)


MemoryError: Unable to allocate 60.9 GiB for an array with shape (2, 314, 146, 91, 1960) and data type float32

In [7]:
fly_superclust_dict.keys()

dict_keys(['inc', 'dec', 'flat', 'total'])

In [8]:
file_path=os.path.join(cluster_dir, 'individual_fly_superclusters_2bin_total.pkl')
with open(file_path, 'wb') as file:
        pickle.dump(fly_superclust_dict, file)

In [ ]:
with open(file_path, 'rb') as file:
        fly_superclust_dict = pickle.load(file)

In [ ]:
fly_num=[226,227,228,234,239,240,241,242,249,250]
for fn in fly_num:
    behave_dic=fly_superclust_dict['flat']
    if fn in list(behave_dic.keys()):
        print(np.count_nonzero(~np.isnan(behave_dic[fn][...,0])))

In [ ]:
fly_superclust_dict['inc'].keys()

In [ ]:
num_vox=[]
for behave in fly_superclust_dict:
    for fly in fly_superclust_dict[behave]:
        count_vox=np.shape(fly_superclust_dict[behave][fly])[-2]
        num_vox.append(count_vox)
max_val=np.max(num_vox)
print(max_val)

In [ ]:
for behave in fly_superclust_dict:
    for fly in fly_superclust_dict[behave]:
        count_vox=np.shape(fly_superclust_dict[behave][fly])[-2]
        if count_vox<max_val:
            pad_val=max_val-count_vox
            print(pad_val)
            matrix=fly_superclust_dict[behave][fly]
            padded_matrix = np.pad(matrix, ((0, 0), (0, pad_val),(0,0)),'constant', constant_values=np.nan)
            fly_superclust_dict[behave][fly]=padded_matrix 

In [ ]:
fly_superclust_dict['inc'][226].shape

In [ ]:
color_map_color_1=np.asarray(['#780000','#C1121F','#E0898F','#F0C4C7','#FFFFFF','#D9E6EF','#B3CDDE','#669BBC', '#003049'])[::-1]
cmap_personal = matplotlib.colors.LinearSegmentedColormap.from_list(
    'cmap', color_map_color_1)

top_color=np.asarray(['#780000','#C1121F','#E0898F','#F0C4C7'])[::-1]
bottom=np.asarray(['#D9E6EF','#B3CDDE','#669BBC', '#003049'])[::-1]
top_half_spect = matplotlib.colors.LinearSegmentedColormap.from_list(
    'top_half_spect', top_color)
bottom_half_spect = matplotlib.colors.LinearSegmentedColormap.from_list(
    'bottom_half_spect', bottom)

# Register the new colormap
plt.register_cmap(cmap=top_half_spect)
plt.register_cmap(cmap=bottom_half_spect)

In [ ]:
event='best_5sec'
n_clusters=500
label_files=[]
cluster_files=[]
for file in os.listdir(cluster_dir):
    print(file)
    if f'_{n_clusters}' in file and 'labels' in file and 'total' in file:
        label_files.append(file)
    elif f'_{n_clusters}_' in file and event in file and 'clusters' in file:
        cluster_files.append(file)

## Load supervox

In [ ]:
cluster_files

In [ ]:
%%time
# flat_cluster_path=os.path.join(cluster_dir,cluster_files[0])
best_cluster_path=os.path.join(cluster_dir,cluster_files[0])
print(best_cluster_path)
with open(best_cluster_path, 'rb') as file:
    best_superclust = pickle.load(file)
    print(best_superclust.keys())
    

In [ ]:
label_files

In [ ]:
%%time
best_label_path=os.path.join(cluster_dir,label_files[0])
print(best_label_path)
giant_total_labels=np.load(best_label_path)
print(giant_total_labels.shape)

In [ ]:
fixed = brainsss.load_fda_meanbrain()
behaviors=list(best_superclust.keys())

In [ ]:
best_superclust[behave].shape[-1]

In [ ]:
# bin_size=5
# downsampled_superclust_dict={}
# for behave in best_superclust:
#     num_bins=int((best_superclust[behave].shape[-1]+1)/bin_size)
#     start=0
#     downsampled_superclust=[]
#     for i in range(num_bins):
#         end=start+bin_size
#         avg_time=np.mean(best_superclust[behave][:,start:end],axis=-1)
#         downsampled_superclust.append(avg_time)
#         start=end
#     downsampled_superclust=np.asarray(downsampled_superclust).T
# #     print(np.shape(downsampled_superclust))
#     downsampled_superclust_dict[behave]=downsampled_superclust
    

In [ ]:
ax_ap=0.1
superclust_dict=best_superclust
fig,axs=plt.subplots(1,3,figsize=(10,10))
axs[0].imshow(superclust_dict['inc'][:,14:41], vmax=0.01,vmin=-0.01)
axs[0].set_aspect(ax_ap)
axs[1].imshow(superclust_dict['dec'][:,14:41],  vmax=0.01,vmin=-0.01)
axs[1].set_aspect(ax_ap)
axs[2].imshow(superclust_dict['flat'][:,14:41],  vmax=0.01,vmin=-0.01)
axs[2].set_aspect(ax_ap)
plt.tight_layout()

In [ ]:
color_map_color_1=np.asarray(['#780000','#C1121F','#E0898F','#F0C4C7','#FFFFFF','#D9E6EF','#B3CDDE','#669BBC', '#003049'])[::-1]
cmap_personal = matplotlib.colors.LinearSegmentedColormap.from_list(
    'cmap', color_map_color_1)

In [ ]:
superclust_dict_z={}
for behavior in best_superclust:
    print(behavior)
    signals_total_z=scipy.stats.zscore(superclust_dict[behavior][:,14:41],axis=1)
    superclust_dict_z[behavior]=signals_total_z

In [ ]:
ax_ap=0.02
vmax=1
vmin=-1
superclust_dict=best_superclust
fig,axs=plt.subplots(1,3,figsize=(15,8))
sns.heatmap(superclust_dict_z['flat'], cmap=cmap_personal,ax=axs[0],yticklabels=False,xticklabels=False,vmax=vmax,vmin=vmin, cbar=False,center=0)
sns.heatmap(superclust_dict_z['inc'], cmap=cmap_personal,ax=axs[1],yticklabels=False,xticklabels=False,vmax=vmax,vmin=vmin, cbar=False,center=0)
sns.heatmap(superclust_dict_z['dec'], cmap=cmap_personal,ax=axs[2],yticklabels=False,xticklabels=False,vmax=vmax,vmin=vmin, cbar=False,center=0)
plt.tight_layout()
cbar = fig.colorbar(axs[-1].collections[0], ax=axs, shrink=0.8)
cbar.set_label('f$_n$', rotation=270, labelpad=20)
cbar.set_ticks([vmax, 0, vmin]) 

# Remove the black border
cbar.outline.set_visible(False)
# plt.savefig(os.path.join(save_dir,'superclust_heatmaps.png'), dpi=300, bbox_inches='tight')

In [ ]:
for fn in fly_num:
    try:
        print(np.count_nonzero(~np.isnan(fly_superclust_dict['dec'][fn][100,:,-1])))
#     print(fly_superclust_dict['dec'][fn][...,4][~np.isnan(fly_superclust_dict['dec'][fn][...,4])])
    except:
        print('not in')

In [ ]:
fly_superclust_dict['inc'].keys()

In [ ]:
fig,axs=plt.subplots(1,3,figsize=(20,8))
for i,be in enumerate(fly_superclust_dict):
    avg=[]
    for fly in fly_superclust_dict[be]:
#         print(fly)
        superclusts=np.nanmean(fly_superclust_dict[be][fly],axis=-2)
        superclusts=np.nan_to_num(superclusts)
        avg.append(superclusts)
#     print(np.shape(avg))
    avg=np.mean(avg,axis=0)
    sns.heatmap(avg, cmap='viridis', cbar=False,ax=axs[i],yticklabels=False,xticklabels=['Before','During'],vmax=0.01,vmin=-0.01)
#     axs[i].set_title(be)
cbar = fig.colorbar(axs[-1].collections[0], ax=axs, shrink=0.8)
cbar.set_label('f$_n$', rotation=270, labelpad=20)

In [ ]:
downsampled_superclust_dict['flat'].shape

In [ ]:
np.shape(list(fly_superclust_dict[behave].values()))

In [ ]:
np.all(np.isnan(fly_superclust_dict['dec'][226][100,:,-1]))

In [ ]:
intervals

In [ ]:
downsampled_superclust_dict={}
for behave in best_superclust:
    cluster_dict=np.asarray(best_superclust[behave])
    downsampled_superclust=[]
    for interval in intervals:
        start=interval[0]
        end=interval[1]
        avg_time=np.mean(cluster_dict[:,start:end],axis=-1)
        downsampled_superclust.append(avg_time)
        start=end
    downsampled_superclust=np.asarray(downsampled_superclust).T
#     print(np.shape(downsampled_superclust))
    downsampled_superclust_dict[behave]=downsampled_superclust

In [ ]:
%%time
anova_dict={}
for behave in fly_superclust_dict:
    anova_dict[behave]={}
    avg_clusts=downsampled_superclust_dict[behave]
    individual_clusts=fly_superclust_dict[behave]
    for i,super_clust in enumerate(avg_clusts):
        anova_dict[behave][i]=[]
        for v, tp in enumerate(super_clust):
            ind_clus=[]
            for fly in individual_clusts:
                ind_clus_nan=individual_clusts[fly][i,:,v]
#                 print(ind_clus_nan)
#                 if np.all(np.isnan(ind_clus_nan))==False:
                ind_clus=np.concatenate((ind_clus,ind_clus_nan))

            anova_dict[behave][i].append(np.asarray(ind_clus))
        anova_dict[behave][i]=np.asarray(anova_dict[behave][i])
#         print(np.shape(anova_dict[behave][i]))

In [ ]:
anova_dict['inc'][150].shape

In [ ]:
np.count_nonzero(~np.isnan(anova_dict['flat'][499][-1]))

In [ ]:
b_1='flat'
b_2='dec'
pval_dict=[]
for clust in range(500):
    a=anova_dict[b_1][clust]
#     print(np.shape(a))
    b=anova_dict[b_2][clust]
#     print(np.shape(b))
    tstat,p_val=scipy.stats.ttest_ind(a,b,axis=-1,nan_policy='omit')
# #     print(p_val)
    pval_dict.append(list(p_val))

In [ ]:
pval_flat=np.asarray(pval_dict).flatten()
rejected, p_corrected, alpha_sidak, alpha_bonf = multipletests(pval_flat, alpha=0.01, method='bonferroni')
pval_correct=p_corrected.reshape(*np.shape(pval_dict))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 20))  # Tall and narrow for 500x5
sns.heatmap(pval_correct, 
            cmap='inferno_r', 
            ax=ax,yticklabels=False, cbar=False,
            xticklabels=['Before','During'],vmax=0.01)
# cbar = fig.colorbar(ax.collections[0], ax=ax)
# cbar.set_label('p-val', rotation=270, labelpad=20)
# cbar.outline.set_visible(False)
# cbar.set_ticks([vmax, 0, vmin]) 
# ax.axhline(y=379, color='w', lw=2)
# plt.savefig(os.path.join(save_dir,f'p_vals_{b_1}_{b_2}.png'), dpi=300, bbox_inches='tight')

In [ ]:
num_clu=50
sort_clust=np.argsort(pval_correct)
bottom_clust=sort_clust[:,:num_clu][0]

In [ ]:
labels_anat=supercluster_labels.reshape(314,146,91)

In [ ]:
labels_anat_sub_total=np.where(np.isin(labels_anat,bottom_clust),labels_anat,np.nan)   
fig,ax=plt.subplots(figsize=(10,10))
ax.imshow(np.nanmax(fixed.numpy(),axis=-1).T,cmap='Greys_r')
ax.imshow(np.nanmax(labels_anat_sub_total,axis=-1).T,cmap='summer')
